# 🔍 Agent 2: Document Extraction Agent

**Purpose:** Extracts structured data from documents based on their type.

| Document Type | Extracted Fields |
|--------------|------------------|
| 📄 Invoice | Invoice #, Date, Vendor, Items, Amounts, Total, Tax |
| 📋 Contract | Parties, Term, Value, Termination Clause |
| 📊 Report | Summary, Key Metrics, Recommendations |
| 📝 Email | Sender, Recipient, Subject, Date, Summary |

**Tech:** Gemini 2.0 Flash via Vertex AI

In [1]:
from google import genai
from google.genai import types
import json

# Initialize Vertex AI client
client = genai.Client(
    vertexai=True,
    project="pwc-agentic-demo",
    location="us-central1"
)

print("✅ Vertex AI client ready for extraction!")

✅ Vertex AI client ready for extraction!


## 🧠 Define Extraction Prompts per Document Type

In [2]:
EXTRACTION_PROMPTS = {
    "Invoice": """Extract the following fields from this invoice. Return as JSON:
{
  "invoice_number": "string",
  "date": "string (YYYY-MM-DD or as-is)",
  "vendor": {
    "name": "string",
    "address": "string"
  },
  "bill_to": {
    "name": "string",
    "address": "string"
  },
  "line_items": [
    {
      "description": "string",
      "quantity": number,
      "rate": "string",
      "amount": "string"
    }
  ],
  "subtotal": "string",
  "tax": "string",
  "total": "string",
  "payment_terms": "string",
  "bank_details": "string"
}

If a field is not found, use null.""",

    "Contract": """Extract the following fields from this contract. Return as JSON:
{
  "contract_title": "string",
  "effective_date": "string",
  "parties": [
    {
      "name": "string",
      "role": "Provider/Client/Both"
    }
  ],
  "scope": "string (brief summary of services)",
  "term": "string (duration)",
  "compensation": "string",
  "key_terms": [
    {
      "clause": "string",
      "summary": "string"
    }
  ],
  "termination_clause": "string"
}

If a field is not found, use null.""",

    "Report": """Extract the following fields from this report. Return as JSON:
{
  "title": "string",
  "period": "string",
  "executive_summary": "string",
  "key_metrics": [
    {
      "metric": "string",
      "value": "string",
      "trend": "string (up/down/stable)"
    }
  ],
  "challenges": ["string"],
  "recommendations": ["string"]
}

If a field is not found, use null.""",

    "Email": """Extract the following fields from this email. Return as JSON:
{
  "from": "string",
  "to": "string",
  "cc": "string or null",
  "subject": "string",
  "date": "string",
  "body_summary": "string (2-3 sentence summary)",
  "action_items": ["string"],
  "sentiment": "string (positive/neutral/negative)"
}

If a field is not found, use null."""
}

print(f"✅ Extraction prompts loaded for: {list(EXTRACTION_PROMPTS.keys())}")

✅ Extraction prompts loaded for: ['Invoice', 'Contract', 'Report', 'Email']


## 🔧 Define the Extraction Function

In [6]:
def extract_fields(text: str, doc_type: str) -> dict:
    """
    Extracts structured fields from a document based on its classified type.
    
    Args:
        text: The document text content
        doc_type: One of 'Invoice', 'Contract', 'Report', 'Email', 'Unknown'
    
    Returns:
        dict: Extracted fields as structured JSON
    """
    if doc_type not in EXTRACTION_PROMPTS:
        return {"error": f"Unknown document type: {doc_type}", "extracted_fields": {}}
    
    prompt = f"""You are a document data extraction expert.

{EXTRACTION_PROMPTS[doc_type]}

Document:
---
{text}
---

Respond ONLY with the JSON object. No additional text."""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0.1,
            response_mime_type="application/json"
        )
    )

    return json.loads(response.text)

## 🧪 Test 1: Extract Fields from Invoice

In [8]:
sample_invoice = """
INVOICE #INV-2024-0042
Date: March 15, 2024

From:
TechCorp Solutions Pvt. Ltd.
123 Innovation Hub, HITEC City, Hyderabad

Bill To:
PricewaterhouseCoopers
5th Floor, Block B, Cyber Towers, Hyderabad

Description                    Qty    Rate      Amount
AI Consulting Services          40    $150/hr    $6,000.00
Cloud Infrastructure Setup       1    $3,500.00   $3,500.00

                              Subtotal:   $9,500.00
                              Tax (18%):  $1,710.00
                              Total:     $11,210.00

Payment Terms: Net 30 days
Bank: HDFC Bank, A/C: 50100012345678
IFSC: HDFC0001234
"""

result = extract_fields(sample_invoice, "Invoice")
print(json.dumps(result, indent=2))

{
  "invoice_number": "INV-2024-0042",
  "date": "March 15, 2024",
  "vendor": {
    "name": "TechCorp Solutions Pvt. Ltd.",
    "address": "123 Innovation Hub, HITEC City, Hyderabad"
  },
  "bill_to": {
    "name": "PricewaterhouseCoopers",
    "address": "5th Floor, Block B, Cyber Towers, Hyderabad"
  },
  "line_items": [
    {
      "description": "AI Consulting Services",
      "quantity": 40,
      "rate": "$150/hr",
      "amount": "$6,000.00"
    },
    {
      "description": "Cloud Infrastructure Setup",
      "quantity": 1,
      "rate": "$3,500.00",
      "amount": "$3,500.00"
    }
  ],
  "subtotal": "$9,500.00",
  "tax": "$1,710.00",
  "total": "$11,210.00",
  "payment_terms": "Net 30 days",
  "bank_details": "HDFC Bank, A/C: 50100012345678 IFSC: HDFC0001234"
}


## 🧪 Test 2: Extract Fields from Contract

In [9]:
sample_contract = """
MASTER SERVICES AGREEMENT

Effective Date: January 1, 2024

Between:
ABC Technology Pvt. Ltd. ("Provider") - CIN: U72200TG2020PTC123456
and
XYZ Consulting ("Client")

1. SCOPE OF SERVICES
Provider shall deliver AI-powered document processing solutions including:
- Document classification and routing
- Automated data extraction
- Validation workflows
as detailed in Exhibit A.

2. TERM
This Agreement shall remain in effect for 12 months from the Effective Date.

3. COMPENSATION
Client shall pay Provider $150,000 annually, payable in quarterly installments of $37,500.

4. CONFIDENTIALITY
Both parties agree to maintain strict confidentiality of all proprietary information
for a period of 2 years after termination.

5. TERMINATION
Either party may terminate with 30 days written notice.
Provider may terminate immediately upon non-payment exceeding 15 days.
"""

result = extract_fields(sample_contract, "Contract")
print(json.dumps(result, indent=2))

{
  "contract_title": "MASTER SERVICES AGREEMENT",
  "effective_date": "January 1, 2024",
  "parties": [
    {
      "name": "ABC Technology Pvt. Ltd.",
      "role": "Provider"
    },
    {
      "name": "XYZ Consulting",
      "role": "Client"
    }
  ],
  "scope": "Provider shall deliver AI-powered document processing solutions including document classification and routing, automated data extraction, and validation workflows.",
  "term": "12 months from the Effective Date",
  "compensation": "$150,000 annually, payable in quarterly installments of $37,500.",
  "key_terms": [
    {
      "clause": "CONFIDENTIALITY",
      "summary": "Both parties agree to maintain strict confidentiality of all proprietary information for a period of 2 years after termination."
    }
  ],
  "termination_clause": "Either party may terminate with 30 days written notice. Provider may terminate immediately upon non-payment exceeding 15 days."
}


## 🧪 Test 3: Extract Fields from Report

In [10]:
sample_report = """
QUARTERLY PERFORMANCE REPORT - Q4 2023

Executive Summary:
Revenue increased by 23% YoY to $4.2M. Client acquisition rate improved
by 15% compared to Q3. Employee retention stands at 92%.

Key Metrics:
- Total Revenue: $4,200,000 (↑ 23% YoY)
- New Clients: 12 (↑ from 8 in Q3)
- Client Satisfaction Score: 4.6/5.0 (→ stable)
- Project Delivery On-Time Rate: 94% (↑ from 91%)

Challenges:
1. Cloud infrastructure costs exceeded budget by 8%
2. Two senior engineers resigned in November

Recommendations:
- Negotiate better cloud pricing with AWS
- Implement retention bonuses for key personnel
- Expand sales team by 3 headcount in Q1 2024
"""

result = extract_fields(sample_report, "Report")
print(json.dumps(result, indent=2))

{
  "title": "QUARTERLY PERFORMANCE REPORT",
  "period": "Q4 2023",
  "executive_summary": "Revenue increased by 23% YoY to $4.2M. Client acquisition rate improved by 15% compared to Q3. Employee retention stands at 92%.",
  "key_metrics": [
    {
      "metric": "Total Revenue",
      "value": "$4,200,000",
      "trend": "up"
    },
    {
      "metric": "New Clients",
      "value": "12",
      "trend": "up"
    },
    {
      "metric": "Client Satisfaction Score",
      "value": "4.6/5.0",
      "trend": "stable"
    },
    {
      "metric": "Project Delivery On-Time Rate",
      "value": "94%",
      "trend": "up"
    }
  ],
  "challenges": [
    "Cloud infrastructure costs exceeded budget by 8%",
    "Two senior engineers resigned in November"
  ],
  "recommendations": [
    "Negotiate better cloud pricing with AWS",
    "Implement retention bonuses for key personnel",
    "Expand sales team by 3 headcount in Q1 2024"
  ]
}


## 📁 Test 4: Extract from PDF File

In [11]:
def extract_from_pdf(pdf_path: str, doc_type: str) -> dict:
    """
    Extracts fields from a PDF document.
    """
    import fitz  # PyMuPDF
    
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    doc.close()
    
    if len(text) > 50000:
        text = text[:50000] + "\n\n[Document truncated...]"
    
    return extract_fields(text, doc_type)

In [12]:
# Test with a PDF (uncomment and provide path)
# result = extract_from_pdf(r"C:\Users\madha\code\pwc-demo\sample_docs\invoice.pdf", "Invoice")
# print(json.dumps(result, indent=2))

## 🔗 End-to-End Test: Classify → Extract

Chain Agent 1 (Classification) with Agent 2 (Extraction)

In [ ]:
# Import classification function from Agent 1
import sys
sys.path.insert(0, r"C:\Users\madha\code\pwc-demo\notebooks")

def classify_document(text: str) -> dict:
    import json
    prompt = f"""You are a document classification expert.
Classify the following document into exactly ONE category:
- Invoice
- Contract
- Report
- Email
- Unknown

Respond in this exact JSON format:
{{"type": "<category>", "confidence": <0.0-1.0>, "reasoning": "<brief explanation>"}}

Document:
---
{text}
---"""
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0.1,
            response_mime_type="application/json"
        )
    )
    return json.loads(response.text)


def process_document(text: str) -> dict:
    """
    Full pipeline: Classify → Extract
    """
    # Step 1: Classify
    classification = classify_document(text)
    doc_type = classification["type"]
    confidence = classification["confidence"]
    
    print(f"📄 Document Type: {doc_type} (confidence: {confidence})")
    
    # Step 2: Extract fields based on type
    if doc_type != "Unknown":
        extracted = extract_fields(text, doc_type)
    else:
        extracted = {"message": "Cannot extract fields from unknown document type"}
    
    return {
        "classification": classification,
        "extracted_fields": extracted
    }

In [ ]:
# Test end-to-end pipeline
sample = """
INVOICE #INV-2024-0042
Date: March 15, 2024
From: TechCorp Solutions Pvt. Ltd.
To: PricewaterhouseCoopers

AI Consulting Services - 40 hrs @ $150/hr = $6,000
Total: $6,000 + 18% tax = $7,080
"""

result = process_document(sample)
print("\n📋 Full Result:")
print(json.dumps(result, indent=2))

## ✅ Extraction Agent - Complete

**What we built:**
- Type-specific extraction prompts for Invoice, Contract, Report, Email
- `extract_fields()` function with structured JSON output
- PDF extraction support
- End-to-end pipeline: Classify → Extract

**Next:** Validation Agent (Agent 3) — validates extracted data against business rules